In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
 
inventory_df = spark.table("supply_chain_opt.inventory")
vendor_logs_df = spark.table("supply_chain_opt.vendor_logs")

In [0]:
vendor_stats = vendor_logs_df.groupBy("vendor_id").agg(
    F.stddev(F.datediff("delivery_date", "order_date")).alias("lead_time_variability")
).fillna(0, subset=["lead_time_variability"])
 
avg_variability = vendor_stats.agg(F.avg("lead_time_variability")).first()[0]
 
inventory_adj = inventory_df.join(vendor_stats, "vendor_id", "left") \
    .withColumn("lead_time_variability", F.coalesce(F.col("lead_time_variability"), F.lit(0.0))) \
    .withColumn(
        "reorder_point_adj",
        F.when(
            F.col("lead_time_variability") > F.lit(avg_variability),
            F.round(F.col("reorder_point") * 1.10).cast("int")
        ).otherwise(F.col("reorder_point"))
    )
 
inventory_adj.write.mode("overwrite").saveAsTable("supply_chain_opt.inventory_adjusted")
 
bumped = inventory_adj.filter(F.col("reorder_point_adj") != F.col("reorder_point")).count()
print(f"Safety stock adjustment applied: {bumped} SKUs (linked to above-average-variability vendors) "
      f"had their reorder point raised 10%. Average lead-time variability across vendors: {avg_variability:.2f} days.")

Safety stock adjustment applied: 46000 SKUs (linked to above-average-variability vendors) had their reorder point raised 10%. Average lead-time variability across vendors: 4.33 days.


In [0]:
inventory_adj = spark.table("supply_chain_opt.inventory_adjusted")
 
reorder_alerts_df = inventory_adj.filter(F.col("current_stock") < F.col("reorder_point_adj")) \
    .withColumn("shortfall_units", F.col("reorder_point_adj") - F.col("current_stock")) \
    .withColumn("replenishment_cost", F.round(F.col("shortfall_units") * F.col("unit_cost"), 2))
 
reorder_alerts_df.write.mode("overwrite").saveAsTable("supply_chain_opt.reorder_alerts")
print(f"Phase 2.2 complete: {reorder_alerts_df.count()} items require reorder "
      f"(using the vendor-adjusted reorder point).")

Phase 2.2 complete: 55838 items require reorder (using the vendor-adjusted reorder point).


In [0]:
display(
    spark.table("supply_chain_opt.reorder_alerts")
    .select("product_name", "current_stock", "reorder_point_adj", "replenishment_cost")
    .orderBy(F.desc("replenishment_cost"))
    .limit(100)
)

product_name,current_stock,reorder_point_adj,replenishment_cost
Item_75905,171,1423,631947.0
Item_95174,147,1416,612330.57
Item_36277,129,1406,577165.69
Item_88845,189,1397,571444.4
Item_10822,117,1261,567561.28
Item_22611,172,1277,557616.15
Item_66349,212,1379,547439.7
Item_71359,364,1438,543207.72
Item_63483,137,1242,539748.3
Item_86550,177,1282,537626.7


In [0]:
abc_df = inventory_adj.withColumn(
    "annual_usage_value", F.round(F.col("daily_demand") * F.col("unit_cost") * 365, 2)
)
 
window_spec = Window.orderBy(F.desc("annual_usage_value"))
abc_df = abc_df.withColumn("total_val", F.sum("annual_usage_value").over(Window.partitionBy())) \
    .withColumn("cum_val", F.sum("annual_usage_value").over(window_spec)) \
    .withColumn("cum_pct", (F.col("cum_val") / F.col("total_val")) * 100)
 
abc_df = abc_df.withColumn(
    "abc_category",
    F.when(F.col("cum_pct") <= 80, "A")
    .when(F.col("cum_pct") <= 95, "B")
    .otherwise("C")
)
 
abc_df.select("product_id", "abc_category", "annual_usage_value") \
    .write.mode("overwrite").saveAsTable("supply_chain_opt.inventory_abc")
print("ABC analysis complete. Categories are now driven by daily_demand x unit_cost, not by reorder_point.")
 

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


ABC analysis complete. Categories are now driven by daily_demand x unit_cost, not by reorder_point.


In [0]:
health_metrics_df = inventory_adj.withColumn(
    "annual_turnover_ratio",
    F.when(F.col("current_stock") != 0,
           F.round((F.col("daily_demand") * 365) / F.col("current_stock"), 2)).otherwise(None)
).withColumn(
    "stock_out_risk_score",
    F.when(F.col("current_stock") != 0,
           F.round((F.col("daily_demand") * F.col("lead_time_days")) / F.col("current_stock"), 2)).otherwise(None)
).withColumn(
    "priority_level",
    F.when(F.col("stock_out_risk_score") > 1.2, "CRITICAL")
    .when(F.col("stock_out_risk_score") > 0.8, "HIGH")
    .otherwise("STABLE")
)
 
health_metrics_df.select("product_id", "annual_turnover_ratio", "stock_out_risk_score", "priority_level") \
    .write.mode("overwrite").saveAsTable("supply_chain_opt.inventory_health")
print("Inventory health metrics recalculated. stock_out_risk_score > 1.0 means expected demand "
      "during lead time exceeds current stock.")

Inventory health metrics recalculated. stock_out_risk_score > 1.0 means expected demand during lead time exceeds current stock.


In [0]:
abc_data = spark.table("supply_chain_opt.inventory_abc")
health_data = spark.table("supply_chain_opt.inventory_health")

spark.sql("DROP TABLE IF EXISTS supply_chain_opt.gold_inventory_master")
 
gold_table = inventory_adj \
    .join(abc_data.select("product_id", "abc_category", "annual_usage_value"), "product_id", "left") \
    .join(health_data.select("product_id", "annual_turnover_ratio", "stock_out_risk_score", "priority_level"), "product_id", "left") \
    .withColumn("is_reorder_required", F.col("current_stock") < F.col("reorder_point_adj")) \
    .withColumn(
        "value_at_risk",
        F.when(F.col("is_reorder_required"),
               F.round((F.col("reorder_point_adj") - F.col("current_stock")) * F.col("unit_cost"), 2))
        .otherwise(0.0)
    )
 
gold_table.write.mode("overwrite").saveAsTable("supply_chain_opt.gold_inventory_master")
print("Gold master table rebuilt on top of the corrected reorder points, ABC values, and risk scores.")
 

Gold master table rebuilt on top of the corrected reorder points, ABC values, and risk scores.
